# Week 8 — Day 2: RAG, FrontierAgent, Ensemble Agent

Building the next two agents in the price-prediction system:
1. A **RAG-based FrontierAgent** that uses GPT-5.1 with retrieved similar-product context (instead of fine-tuning)
2. An **EnsembleAgent** that combines GPT-5.1 RAG, the fine-tuned Llama specialist, and a deep neural network with weighted averaging

Backed by a 400,000-product Chroma vector store built from the Amazon dataset prepared in week 6.

## Setup

In [ ]:
import os
import re
import logging
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [ ]:
load_dotenv(override=True)
DB = "products_vectorstore"

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

In [ ]:
LITE_MODE = False
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## Build the Chroma vector store

400,000 products from the training dataset, encoded with `all-MiniLM-L6-v2` (free, fast, runs locally — 384-dim vectors).

In [ ]:
client = chromadb.PersistentClient(path=DB)

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
vector = encoder.encode(["Test sentence to confirm encoder is working"])[0]
print(vector.shape)
vector

In [ ]:
# Populate the collection in batches of 1,000.
# This takes ~30 minutes on a GPU. Use LITE_MODE=True for a faster run.
collection_name = "products"
existing_collection_names = [c.name for c in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i:i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i:i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)][:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

## Visualize the embedding space

10,000 products is safe for any machine; pushing toward 800k can crash if you're tight on RAM.

In [ ]:
MAXIMUM_DATAPOINTS = 10_000

In [ ]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics',
              'Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green', 'purple', 'red']

In [ ]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [m['category'] for m in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200, height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])
fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200, height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)
fig.show()

## RAG retrieval helpers

In [ ]:
test[0]

In [ ]:
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
find_similars(test[0])

## Build context for the LLM with retrieved similar products

In [ ]:
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

In [ ]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

## RAG inference with GPT-5.1

In [ ]:
def gpt_5_1_rag(item):
    documents, prices = find_similars(item)
    response = completion(
        model="gpt-5.1",
        messages=messages_for(item, documents, prices),
        reasoning_effort="none",
        seed=42
    )
    return response.choices[0].message.content

In [ ]:
test[0].price

In [ ]:
gpt_5_1_rag(test[0])

In [ ]:
evaluate(gpt_5_1_rag, test)

## Bring in the SpecialistAgent (deployed in day 1) and the Deep Neural Network

In [ ]:
import modal

Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

def specialist(item):
    return pricer.price.remote(item.summary)

In [ ]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Deep Neural Network agent

Drop the trained weights file `deep_neural_network.pth` (from week 6) into this directory before running the next cell.

In [ ]:
from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

def deep_neural_network(item):
    return runner.inference(item.summary)

## Simple ensemble — weighted average of all three predictors

In [ ]:
def ensemble(item):
    price1 = get_price(gpt_5_1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1

In [ ]:
evaluate(ensemble, test)

## Wrap everything in agent classes

These are the production versions used by the autonomous framework in days 4 and 5.

In [ ]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")